# Extração de Features
## Parte 1: Segmentação — Entregável 5

### O que esta parte faz

A partir daqui começa a **Fase B — Extração Propriamente Dita**, o trabalho principal
para a disciplina de reconhecimento de padrões.

Esta parte resolve quatro problemas em sequência:

1. **Estratégia de janelamento**: janelas de 1 segundo com 50% de sobreposição
2. **Aplicar máscara de qualidade**: SQI inline equivalente ao PP5
3. **Segmentar por task**: cada janela herda um label único; janelas de transição são descartadas
4. **Validar a segmentação**: variância intra- e inter-janela

### Nota sobre a fonte de dados

O pipeline de pré-processamento (PP1→PP4) produziu `normalized.parquet` com sinais
devidamente filtrados e normalizados, mas **sem a coluna `label`** e com `task_id='unknown'`
em todo o dataset. Isso foi uma **decisão de design do PP1**: a tabela bruta
(`full_table.parquet`) foi montada apenas com os sinais EEG/IMU brutos, enquanto os
labels verdadeiros residem nos arquivos `bronze/filtered_parquet/`, gerados pelo notebook
`bronze_filtrada.ipynb` a partir de `.txt` pré-filtrados do dataset Mendeley.

Como reconhecimento de padrões **exige labels**, esta parte usa os arquivos
`bronze/filtered_parquet/` como fonte primária de sinal — eles já estão a 500 Hz,
contêm `label` (0=normal, 1=FoG) e `task_id` por amostra. Aplicamos Z-score por paciente
inline (equivalente ao PP4) e SQI inline (equivalente ao PP5).

---
### Entradas
- `data/bronze/filtered_parquet/`: sinais filtrados com labels, 500 Hz, por paciente/task
- `data/silver/sqi_mask.parquet`: referência dos thresholds SQI validados no PP5

### Saídas
- `data/silver/segments.npz`: array 3D `X` (n_janelas, 500, n_canais) + vetor `y` (n_janelas,)
- `data/silver/windows_metadata.parquet`: metadados de cada janela (paciente, task, posição, label)
- `data/silver/segmentation_report.txt`: relatório textual
---

## 0. Imports e constantes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import stats as sts

# ── Caminhos ──────────────────────────────────────────────────────────────────
ROOT          = Path("..")
BRONZE_PATH   = ROOT / "data" / "bronze" / "filtered_parquet"
SQI_MASK_PATH = ROOT / "data" / "silver" / "sqi_mask.parquet"
SEGMENTS_PATH = ROOT / "data" / "silver" / "segments.npz"
META_PATH     = ROOT / "data" / "silver" / "windows_metadata.parquet"
REPORT_PATH   = ROOT / "data" / "silver" / "segmentation_report.txt"

# ── Parâmetros de janelamento ─────────────────────────────────────────────────
FS         = 500          # frequência de amostragem (Hz)
WINDOW_S   = 1            # duração da janela em segundos
WINDOW_N   = WINDOW_S * FS   # = 500 amostras por janela
HOP_N      = WINDOW_N // 2   # = 250 amostras (50% de sobreposição)

# ── Thresholds SQI (idênticos ao PP5) ────────────────────────────────────────
KURT_THRESH = 10.0   # kurtose > 10  → spike / artefato impulsivo
PP_THRESH   = 10.0   # pp/std   > 10 → outlier de amplitude

# ── Canais de sinal (igual ao PP5) ───────────────────────────────────────────
EEG_COLS = [
    'EEG-FP1','EEG-FP2','EEG-F3','EEG-F4','EEG-C3','EEG-C4',
    'EEG-P3','EEG-P4','EEG-O1','EEG-O2','EEG-F7','EEG-F8',
    'EEG-P7','EEG-P8','EEG-FZ','EEG-CZ','EEG-PZ',
    'EEG-FC1','EEG-FC2','EEG-CP1','EEG-CP2',
    'EEG-FC5','EEG-FC6','EEG-CP5','EEG-CP6'
]
BIO_COLS       = ['EMG-RTA', 'EMG-LTA', 'EMG-RGS', 'IO', 'ECG']
SENSOR_NAMES   = ['LShank', 'RShank', 'Waist', 'Arm']
IMU_SINAL_SUF  = ['ACCX','ACCY','ACCZ','GYRO-X','GYRO-Y','GYRO-Z']
IMU_COLS_SINAL = [f"{s}-{ax}" for s in SENSOR_NAMES for ax in IMU_SINAL_SUF]
SIGNAL_COLS    = EEG_COLS + BIO_COLS + IMU_COLS_SINAL   # 25 + 5 + 24 = 54 canais

# Canais-chave para validação de qualidade (mesmo critério do PP5)
CANAIS_CHAVE = ['EEG-C3', 'EEG-FP1', 'EMG-RTA']

print(f"Janela : {WINDOW_S}s = {WINDOW_N} amostras @ {FS}Hz")
print(f"Hop    : {HOP_N} amostras → sobreposição de 50%")
print(f"Canais de sinal: {len(SIGNAL_COLS)} ({len(EEG_COLS)} EEG + {len(BIO_COLS)} Bio + {len(IMU_COLS_SINAL)} IMU)")

## 1. Estratégia de Janelamento

### Por que 1 segundo?

A janela de **1 segundo (500 amostras a 500 Hz)** é o padrão para tarefas de
classificação motora com EEG/EMG por três razões fisiológicas:

| Razão | Detalhe |
|-------|----------|
| Captura banda delta completa | Delta (1–4 Hz): 1 s contém ≥ 1 ciclo completo |
| Inclui alpha e beta | Alpha (8–13 Hz): 8 a 13 ciclos por janela |
| Resolução temporal | FoG tem onset entre 0.5 e 3 s → janelas de 1 s capturam início |

Janelas mais curtas (< 0.5 s) não capturam as ondas delta completas.
Janelas mais longas (> 2 s) diluem o onset do FoG em contexto não-FoG.

### Por que 50% de sobreposição?

O avanço de **250 amostras (0.5 s)** entre janelas consecutivas garante:
- **Resolução temporal suficiente**: eventos de 0.5 s são capturados por pelo menos 2 janelas
- **Independência razoável**: janelas adjacentes compartilham 50% das amostras, o que é
  o padrão aceito para não criar janelas totalmente dependentes (que seria 99% de overlap)
  nem perder eventos rápidos (que seria 0% de overlap)
- **Aumento do dataset**: duplica aproximadamente o número de janelas disponíveis para treino

### Justificativa numérica

```
Duração total do bronze dataset: ~3.4 horas de sinal multimodal
Janelas sem overlap (1s):        ~12.240 janelas antes de filtro
Janelas com 50% overlap (0.5s):  ~24.480 janelas antes de filtro
Estimativa pós-SQI (85% ok):     ~20.808 janelas aprovadas
```

## 2. Carregar Máscara SQI de Referência

Carregamos a `sqi_mask.parquet` do PP5 para referência dos thresholds usados.
Como esta parte usa os dados bronze (e não o `normalized.parquet` usado no PP5),
a máscara não pode ser aplicada diretamente — os índices `n_inicio`/`n_fim` da máscara
referem-se a posições no `normalized.parquet`. Aplicamos os **mesmos critérios SQI
inline** durante a segmentação.

In [ ]:
sqi_mask = pd.read_parquet(SQI_MASK_PATH)

print("Máscara SQI (PP5) — referência de thresholds:")
print(f"  Arquivo         : {SQI_MASK_PATH.name}")
print(f"  Shape           : {sqi_mask.shape}")
print(f"  Janelas PP5     : {len(sqi_mask):,} (5s, no normalized.parquet)")
approved = sqi_mask['aprovada_para_extracao'].sum()
print(f"  Aprovadas no PP5: {approved:,} / {len(sqi_mask):,}  ({approved/len(sqi_mask)*100:.1f}%)")
print()
print("Critérios adotados para esta parte (idênticos ao PP5):")
print(f"  Kurtose  > {KURT_THRESH}  → janela rejeitada")
print(f"  pp/std   > {PP_THRESH}  → janela rejeitada")
print(f"  Canais-chave para validação global: {CANAIS_CHAVE}")
print()
print("Nota: os índices n_inicio/n_fim da máscara referem-se ao normalized.parquet")
print("      (fonte diferente). SQI será recomputado inline sobre os dados bronze.")

## 3. Funções de SQI Inline e Normalização

Definimos as funções que serão chamadas durante o loop de segmentação.

**SQI inline**: para cada janela de 1 segundo, calculamos kurtose e razão pico-a-pico/std
em cada canal. Uma janela é **rejeitada** se qualquer canal-chave (EEG-C3, EEG-FP1, EMG-RTA)
violar um threshold. Isso é diretamente equivalente ao critério `janela_global_valida` do PP5.

**Normalização Z-score**: para cada paciente, calculamos média e desvio padrão por canal
sobre todos os dados daquele paciente (union dos tasks), depois subtraímos a média e
dividimos pelo desvio. Isso replica exatamente o que o PP4 fez no `normalized.parquet`.

In [ ]:
def calcular_kurtose(sinal: np.ndarray) -> float:
    if np.std(sinal) < 1e-10:
        return 0.0
    return float(sts.kurtosis(sinal, fisher=True, bias=False))


def calcular_pp_ratio(sinal: np.ndarray) -> float:
    std = np.std(sinal)
    if std < 1e-10:
        return 0.0
    return float((sinal.max() - sinal.min()) / std)


def janela_valida_sqi(janela: np.ndarray, cols_presentes: list) -> bool:
    """
    Verifica se uma janela passa nos critérios SQI dos canais-chave.

    Replica o critério 'janela_global_valida' do PP5:
    todos os canais-chave devem ter kurtose <= KURT_THRESH E pp/std <= PP_THRESH.

    janela: array (WINDOW_N, n_canais) — já Z-scored
    cols_presentes: lista de nomes de canais na ordem das colunas da janela
    """
    for col in CANAIS_CHAVE:
        if col not in cols_presentes:
            continue
        idx = cols_presentes.index(col)
        sinal = janela[:, idx]

        # Canal ausente (sensor NaN neste paciente) → não bloqueia
        if np.all(np.isnan(sinal)):
            continue

        sinal_limpo = pd.Series(sinal).interpolate(
            method='linear', limit_direction='both').values

        if calcular_kurtose(sinal_limpo) > KURT_THRESH:
            return False
        if calcular_pp_ratio(sinal_limpo) > PP_THRESH:
            return False

    return True


def normalizar_zscore(df_sinal: pd.DataFrame, signal_cols: list) -> pd.DataFrame:
    """
    Z-score por canal: (x - mu) / sigma, calculados sobre todo o paciente.
    Canais 100% NaN são preservados como NaN (sensor ausente para este paciente).
    Aplica clip em ±5σ após normalização (igual ao PP4).
    """
    df_out = df_sinal.copy()
    for col in signal_cols:
        if col not in df_out.columns:
            continue
        vals = df_out[col].values.astype(float)
        if np.isnan(vals).all():
            continue
        mu  = np.nanmean(vals)
        sig = np.nanstd(vals)
        if sig < 1e-10:
            df_out[col] = 0.0
        else:
            normed = np.clip((vals - mu) / sig, -5.0, 5.0)
            df_out[col] = normed.astype(np.float32)
    return df_out


def carregar_bronze_paciente(pid: str) -> pd.DataFrame:
    """
    Carrega todos os task_*.parquet de um paciente e concatena em ordem.
    Trata o caso especial do paciente 008 (2 sessões em subpastas 1/ e 2/).
    Sempre inclui coluna 'session' para que o loop de segmentação possa
    agrupar por (session, task_id) e evitar descontinuidades cross-sessão.
    """
    pid_dir = BRONZE_PATH / pid
    if not pid_dir.exists():
        return pd.DataFrame()

    # Detecta se tem subpastas de sessão (paciente 008)
    subdirs = sorted([d for d in pid_dir.iterdir() if d.is_dir()])

    partes = []
    if subdirs:
        for sess_dir in subdirs:
            for t_file in sorted(sess_dir.glob('task_*.parquet')):
                df_t = pd.read_parquet(t_file)
                df_t['task_id'] = t_file.stem
                df_t['session'] = sess_dir.name   # '1' ou '2'
                partes.append(df_t)
    else:
        for t_file in sorted(pid_dir.glob('task_*.parquet')):
            df_t = pd.read_parquet(t_file)
            df_t['task_id'] = t_file.stem
            df_t['session'] = '1'   # sessão única
            partes.append(df_t)

    if not partes:
        return pd.DataFrame()

    return pd.concat(partes, ignore_index=True)


print("Funções definidas:")
print("  calcular_kurtose()         → kurtose de Fisher")
print("  calcular_pp_ratio()        → razão pico-a-pico / std")
print("  janela_valida_sqi()        → verifica canais-chave contra thresholds")
print("  normalizar_zscore()        → Z-score + clip ±5σ por canal")
print("  carregar_bronze_paciente() → carrega e concatena tasks")
print("    inclui coluna 'session' para tratar pac. 008 (2 sessões) corretamente")

## 4. Loop de Segmentação

Para cada paciente:
1. Carrega dados bronze (com labels e task_id por amostra)
2. Aplica Z-score normalization por paciente
3. Para cada task, gera janelas de 500 amostras com hop de 250
4. Descarta janelas de transição (que cruzam fronteira entre tasks)
5. Filtra janelas via SQI nos canais-chave
6. Armazena janela aprovada com seu label (voto majoritário dentro da janela)

**Label da janela**: usamos voto majoritário — o label (0 ou 1) que aparece no maior
número de amostras da janela. Dentro de uma task, o label pode transitar de 0 → 1
(onset de FoG) ou 1 → 0 (offset de FoG). Uma janela com 60% das amostras em label=1
é classificada como FoG. Isso é mais robusto que usar apenas a primeira amostra.

In [ ]:
print("Iniciando loop de segmentação...")
print("=" * 65)

all_windows = []    # cada elemento: array (WINDOW_N, n_canais)
all_labels  = []    # float: 0.0, 1.0, ou NaN se label desconhecido
meta_rows   = []    # dict com metadados por janela

cols_presentes_global = None  # será definido na primeira iteração
n_canais_global = None

# Rastreia janelas possíveis (sem SQI) por paciente — evita re-carregar depois
n_possiveis_por_pid = {}

pacientes = sorted([p.name for p in BRONZE_PATH.iterdir() if p.is_dir()])

for pid in pacientes:
    df_pid = carregar_bronze_paciente(pid)

    if df_pid.empty:
        print(f"  {pid}: sem dados bronze, pulando")
        continue

    # Canais de sinal presentes para este paciente
    sig_cols_pid = [c for c in SIGNAL_COLS if c in df_pid.columns]

    if cols_presentes_global is None:
        cols_presentes_global = sig_cols_pid
        n_canais_global = len(cols_presentes_global)

    # 1. Z-score por paciente (sobre todos os dados concatenados)
    df_norm = normalizar_zscore(df_pid, sig_cols_pid)

    n_janelas_pid    = 0
    n_rejeitadas_sqi = 0
    n_sem_label      = 0
    n_possiveis_pid  = 0

    # Agrupa por (session, task_id) para evitar descontinuidade cross-sessão.
    # Paciente 008 tem 2 sessões com tasks de mesmo nome — sem este agrupamento
    # os dados das duas sessões seriam concatenados e janelas cruzariam a fronteira.
    grupos = df_norm.groupby(['session', 'task_id'], sort=True)

    for (sessao, task), df_task in grupos:
        df_task = df_task.reset_index(drop=True)
        n_task  = len(df_task)

        if n_task < WINDOW_N:
            continue  # task muito curta para uma janela sequer

        # Conta janelas possíveis nesta task (sem filtro SQI)
        n_possiveis_pid += (n_task - WINDOW_N) // HOP_N + 1

        sinal_arr  = df_task[sig_cols_pid].values.astype(np.float32)
        labels_arr = df_task['label'].values if 'label' in df_task.columns else np.full(n_task, np.nan)

        # 2. Deslizamento de janelas dentro da task
        n_inicio = 0
        while n_inicio + WINDOW_N <= n_task:
            n_fim  = n_inicio + WINDOW_N
            janela = sinal_arr[n_inicio:n_fim]        # (500, n_canais)
            labels_jan = labels_arr[n_inicio:n_fim]   # (500,)

            # --- Label: voto majoritário ---
            labels_validos = labels_jan[~np.isnan(labels_jan)]
            if len(labels_validos) == 0:
                label_jan = np.nan
                n_sem_label += 1
            else:
                vals, counts = np.unique(labels_validos.astype(int), return_counts=True)
                label_jan = float(vals[np.argmax(counts)])

            # --- Filtro SQI nos canais-chave ---
            if not janela_valida_sqi(janela, sig_cols_pid):
                n_rejeitadas_sqi += 1
                n_inicio += HOP_N
                continue

            # --- Aprovada ---
            # Alinhar canais para shape global consistente
            if sig_cols_pid != cols_presentes_global:
                janela_global = np.full((WINDOW_N, n_canais_global), np.nan, dtype=np.float32)
                for i_col, col in enumerate(sig_cols_pid):
                    if col in cols_presentes_global:
                        i_global = cols_presentes_global.index(col)
                        janela_global[:, i_global] = janela[:, i_col]
                janela = janela_global

            all_windows.append(janela)
            all_labels.append(label_jan)
            meta_rows.append({
                'patient_id' : pid,
                'session'    : sessao,
                'task_id'    : task,
                'n_inicio'   : n_inicio,
                'n_fim'      : n_fim,
                'label'      : label_jan,
                'janela_idx' : n_janelas_pid,
            })
            n_janelas_pid += 1

            n_inicio += HOP_N

    n_possiveis_por_pid[pid] = n_possiveis_pid
    print(f"  {pid}: {n_janelas_pid:>5,} janelas aprovadas  "
          f"| SQI rejeitou {n_rejeitadas_sqi:>4,}  "
          f"| sem label {n_sem_label:>3,}")

print()
print(f"Total de janelas aprovadas: {len(all_windows):,}")
print(f"Canais no array 3D        : {n_canais_global}")

## 5. Construir Array 3D e Vetor de Labels

In [ ]:
print("Construindo array 3D...")
X = np.stack(all_windows, axis=0)   # (n_janelas, 500, n_canais)
y = np.array(all_labels, dtype=np.float32)  # (n_janelas,)

df_meta = pd.DataFrame(meta_rows)

print(f"X shape: {X.shape}  dtype={X.dtype}")
print(f"  n_janelas = {X.shape[0]:,}")
print(f"  n_amostras = {X.shape[1]} (1 segundo @ 500Hz)")
print(f"  n_canais = {X.shape[2]} ({len(EEG_COLS)} EEG + {len(BIO_COLS)} Bio + {len(IMU_COLS_SINAL)} IMU)")
print(f"  Memória: {X.nbytes / 1e6:.1f} MB")
print()
print(f"y shape: {y.shape}  dtype={y.dtype}")
n_fog    = int(np.nansum(y == 1.0))
n_normal = int(np.nansum(y == 0.0))
n_nan    = int(np.isnan(y).sum())
print(f"  label=0 (normal): {n_normal:,}  ({n_normal/len(y)*100:.1f}%)")
print(f"  label=1 (FoG)   : {n_fog:,}  ({n_fog/len(y)*100:.1f}%)")
print(f"  label=NaN       : {n_nan:,}  ({n_nan/len(y)*100:.1f}%)")
print()
print("Metadados:")
print(df_meta.head(5).to_string(index=False))

## 6. Resumo por Paciente

In [ ]:
print("Resumo por paciente:")
print("-" * 70)
print(f"  {'Paciente':<12} {'Total':>8} {'Normal':>8} {'FoG':>8} {'NaN':>6} {'%FoG':>8}")
print("  " + "-" * 64)

for pid in sorted(df_meta['patient_id'].unique()):
    sub = df_meta[df_meta['patient_id'] == pid]
    tot  = len(sub)
    fog  = int((sub['label'] == 1.0).sum())
    norm = int((sub['label'] == 0.0).sum())
    nan  = int(sub['label'].isna().sum())
    pct  = fog / tot * 100 if tot > 0 else 0
    print(f"  {pid:<12} {tot:>8,} {norm:>8,} {fog:>8,} {nan:>6,} {pct:>7.1f}%")

print("  " + "-" * 64)
print(f"  {'TOTAL':<12} {len(df_meta):>8,} {n_normal:>8,} {n_fog:>8,} {n_nan:>6,} {n_fog/len(y)*100:>7.1f}%")

## 7. Visualização — Distribuição de Janelas por Paciente e Label

In [ ]:
pids = sorted(df_meta['patient_id'].unique())
n_normal_por_pid = [int((df_meta[df_meta['patient_id']==p]['label'] == 0.0).sum()) for p in pids]
n_fog_por_pid    = [int((df_meta[df_meta['patient_id']==p]['label'] == 1.0).sum()) for p in pids]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Stacked bar: Normal vs FoG por paciente ---
ax = axes[0]
x = np.arange(len(pids))
ax.bar(x, n_normal_por_pid, label='Normal (0)', color='steelblue', alpha=0.85)
ax.bar(x, n_fog_por_pid, bottom=n_normal_por_pid, label='FoG (1)', color='tomato', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(pids, rotation=45, ha='right')
ax.set_xlabel('Paciente')
ax.set_ylabel('Número de janelas')
ax.set_title('Distribuição de janelas por paciente e label')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# --- Pizza: proporção global ---
ax2 = axes[1]
sizes  = [n_normal, n_fog]
labels_pie = [f'Normal (0)\n{n_normal:,}', f'FoG (1)\n{n_fog:,}']
colors = ['steelblue', 'tomato']
ax2.pie(sizes, labels=labels_pie, colors=colors, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 11})
ax2.set_title(f'Proporção global (total = {n_normal+n_fog:,} janelas labeled)')

plt.suptitle(f'Segmentação — {len(all_windows):,} janelas de 1s @ 50% overlap', fontsize=13)
plt.tight_layout()
plt.show()

print()
print("Interpretação:")
print("  Classes desbalanceadas é esperado: FoG é um evento episódico, não contínuo.")
print("  No treinamento do classificador, usar técnicas de balanceamento (SMOTE, class_weight).")

## 8. Exemplos Visuais de Janelas

Plotamos 3 janelas: uma normal (label=0), uma de FoG (label=1), e uma janela
do canal EEG-C3 de cada classe para comparação visual.

In [ ]:
t = np.linspace(0, WINDOW_S, WINDOW_N)

# Buscar índices de janelas com label definido
idx_normal = np.where(y == 0.0)[0]
idx_fog    = np.where(y == 1.0)[0]

canais_plot = ['EEG-C3', 'EMG-RTA', 'LShank-ACCZ']

fig, axes = plt.subplots(len(canais_plot), 2, figsize=(14, 4 * len(canais_plot)))
fig.suptitle('Exemplos de janelas: Normal (esq.) vs FoG (dir.)', fontsize=13)

for row_ax, canal in enumerate(canais_plot):
    if canal not in cols_presentes_global:
        for ax in axes[row_ax]:
            ax.text(0.5, 0.5, f'{canal}\n(ausente)', ha='center', va='center',
                    transform=ax.transAxes, color='gray')
        continue

    i_canal = cols_presentes_global.index(canal)

    for col_ax, (idx_arr, cor, titulo_label) in enumerate([
        (idx_normal, 'steelblue', 'Normal (label=0)'),
        (idx_fog,    'tomato',    'FoG (label=1)'),
    ]):
        ax = axes[row_ax, col_ax]

        if len(idx_arr) == 0:
            ax.text(0.5, 0.5, 'Sem janelas\ndesta classe', ha='center', va='center',
                    transform=ax.transAxes, color='gray')
            ax.set_title(f'{canal} — {titulo_label}')
            continue

        # Pegar a janela mediana (ordenada por índice) para ser representativa
        idx_exemplo = idx_arr[len(idx_arr) // 2]
        sinal = X[idx_exemplo, :, i_canal]

        ax.plot(t, sinal, lw=0.8, color=cor, alpha=0.9)
        ax.set_title(f'{canal} — {titulo_label}\njanela idx={idx_exemplo}', fontsize=9)
        ax.set_xlabel('Tempo (s)')
        ax.set_ylabel('Amplitude (Z-score)')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Passo 4 — Validação da Segmentação

### 9.1 Variância intra-janela

Para cada janela, calculamos o **desvio padrão interno** (std das 500 amostras
dentro da janela) para cada canal.

> **O que esperamos**: após Z-score global por paciente, o std intra-janela mediano
> deve ser próximo de 1.0 para sinais com variância relativamente estacionária.
> Std ≈ 0 indica segmento constante (sensor travado ou canal zerado).
> Std >> 2 indica janela com artefato de amplitude persistente.

### 9.2 Variância inter-janela

Para cada canal, calculamos a **variância das médias** de cada janela.

> **O que esperamos**: para sinais Z-scored, a média de cada janela deve flutuar
> em torno de zero. A variância das médias deve ser pequena (< 0.1) indicando
> que o baseline global está estável e não há drift de longo prazo.

In [ ]:
print("Calculando variâncias intra- e inter-janela...")

# Variância intra-janela: std de cada janela por canal → shape (n_janelas, n_canais)
std_intra = np.nanstd(X, axis=1)   # std ao longo das 500 amostras

# Variância inter-janela: variance da média de cada janela por canal → shape (n_canais,)
mean_por_janela = np.nanmean(X, axis=1)   # média de cada janela: (n_janelas, n_canais)
var_inter       = np.nanvar(mean_por_janela, axis=0)  # variância das médias: (n_canais,)

print()
print("=== Variância Intra-Janela (std por janela por canal) ===")
print(f"  Mediana global do std intra-janela: {np.nanmedian(std_intra):.4f}")
print(f"  (esperado ≈ 1.0 para dado Z-scored com estacionaridade moderada)")
print()

# Janelas com std ≈ 0 em qualquer canal-chave (sensor travado)
n_travadas = 0
for col in CANAIS_CHAVE:
    if col not in cols_presentes_global:
        continue
    i_col = cols_presentes_global.index(col)
    n_trav = int((std_intra[:, i_col] < 1e-6).sum())
    if n_trav > 0:
        print(f"  ATENÇÃO: {n_trav} janelas com std≈0 em {col} (sensor travado?)")
        n_travadas += n_trav

if n_travadas == 0:
    print("  Nenhuma janela com desvio padrão interno zero nos canais-chave ✅")

print()
print("=== Variância Inter-Janela (var das médias por canal) ===")
print(f"  Mediana global da var inter-janela: {np.nanmedian(var_inter):.4f}")
print(f"  (esperado < 0.1 para dado Z-scored sem drift de longo prazo)")
n_altos = int((var_inter > 0.1).sum())
if n_altos > 0:
    print(f"  {n_altos} canais com var_inter > 0.1:")
    for i, (col, v) in enumerate(zip(cols_presentes_global, var_inter)):
        if v > 0.1:
            print(f"    {col}: {v:.4f}")
else:
    print("  Todos os canais com var_inter < 0.1 ✅")

In [ ]:
# Visualização da validação
tipos_canal = []
for col in cols_presentes_global:
    if col.startswith('EEG'):
        tipos_canal.append('EEG')
    elif col.startswith('EMG') or col in ('IO', 'ECG'):
        tipos_canal.append('Bio')
    else:
        tipos_canal.append('IMU')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Histograma do std intra-janela (mediana por canal) ---
ax = axes[0]
mediana_std_por_canal = np.nanmedian(std_intra, axis=0)
cores_tipo = {'EEG': 'steelblue', 'Bio': 'seagreen', 'IMU': 'darkorange'}
for tipo in ['EEG', 'Bio', 'IMU']:
    idx_tipo = [i for i, t in enumerate(tipos_canal) if t == tipo]
    ax.hist(mediana_std_por_canal[idx_tipo], bins=20, alpha=0.7,
            label=tipo, color=cores_tipo[tipo])
ax.axvline(1.0, color='red', ls='--', lw=2, label='Esperado ≈ 1')
ax.set_xlabel('Mediana do std intra-janela (por canal)')
ax.set_ylabel('Número de canais')
ax.set_title('Variância Intra-Janela')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Histograma da variância inter-janela ---
ax2 = axes[1]
for tipo in ['EEG', 'Bio', 'IMU']:
    idx_tipo = [i for i, t in enumerate(tipos_canal) if t == tipo]
    ax2.hist(var_inter[idx_tipo], bins=20, alpha=0.7,
             label=tipo, color=cores_tipo[tipo])
ax2.axvline(0.1, color='red', ls='--', lw=2, label='Limite < 0.1')
ax2.set_xlabel('Variância das médias inter-janela (por canal)')
ax2.set_ylabel('Número de canais')
ax2.set_title('Variância Inter-Janela')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# --- Distribuição do std intra-janela para EEG-C3 especificamente ---
ax3 = axes[2]
if 'EEG-C3' in cols_presentes_global:
    i_c3 = cols_presentes_global.index('EEG-C3')
    std_c3_normal = std_intra[y == 0.0, i_c3] if (y == 0.0).any() else np.array([])
    std_c3_fog    = std_intra[y == 1.0, i_c3] if (y == 1.0).any() else np.array([])
    if len(std_c3_normal) > 0:
        ax3.hist(std_c3_normal, bins=30, alpha=0.7, color='steelblue',
                 label=f'Normal (n={len(std_c3_normal):,})')
    if len(std_c3_fog) > 0:
        ax3.hist(std_c3_fog,    bins=30, alpha=0.7, color='tomato',
                 label=f'FoG (n={len(std_c3_fog):,})')
    ax3.set_xlabel('std intra-janela (EEG-C3 Z-scored)')
    ax3.set_title('EEG-C3: std por classe')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'EEG-C3 não disponível', ha='center', va='center',
             transform=ax3.transAxes)

plt.suptitle('Validação da Segmentação — Variâncias Intra e Inter Janela', fontsize=12)
plt.tight_layout()
plt.show()

print()
print("Interpretação:")
print("  Intra-janela: pico próximo de 1.0 = normalização consistente ✅")
print("  Inter-janela: valores abaixo de 0.1 = baseline estável ✅")
print("  EEG-C3 Normal vs FoG: diferença de variância pode ser uma feature útil.")

## 10. Percentual de Dados Rejeitados por Paciente

In [ ]:
# Taxa de aprovação usando n_possiveis_por_pid calculado no loop principal
print("Percentual de dados rejeitados por paciente:")
print("-" * 65)
print(f"  {'Paciente':<12} {'Possíveis':>12} {'Aprovadas':>10} {'Rejeitadas':>11} {'%Aprovado':>11}")
print("  " + "-" * 61)

total_possiveis = 0
total_aprovadas = 0
for pid in sorted(df_meta['patient_id'].unique()):
    n_aprovadas  = len(df_meta[df_meta['patient_id'] == pid])
    n_possiveis  = n_possiveis_por_pid.get(pid, 0)
    n_rejeitadas = n_possiveis - n_aprovadas
    pct = n_aprovadas / n_possiveis * 100 if n_possiveis > 0 else 0
    emoji = '✅' if pct >= 80 else ('🟡' if pct >= 60 else '🔴')
    print(f"  {pid:<12} {n_possiveis:>12,} {n_aprovadas:>10,} {n_rejeitadas:>11,} {pct:>10.1f}%  {emoji}")
    total_possiveis += n_possiveis
    total_aprovadas += n_aprovadas

print("  " + "-" * 61)
pct_global = total_aprovadas / total_possiveis * 100 if total_possiveis > 0 else 0
print(f"  {'TOTAL':<12} {total_possiveis:>12,} {total_aprovadas:>10,} {total_possiveis-total_aprovadas:>11,} {pct_global:>10.1f}%")
print()
print(f"Nota: taxa de rejeição SQI = {100-pct_global:.1f}%")
print("      Critério: kurtose ou pp/std nos canais-chave (EEG-C3, EEG-FP1, EMG-RTA).")
print("      Canais IMU com alta kurtose (detectados no PP5) não afetam esta taxa.")

## 11. Salvar Resultados

In [ ]:
print("Salvando resultados...")

# ── Array 3D + labels (comprimido) ───────────────────────────────────────────
np.savez_compressed(
    SEGMENTS_PATH,
    X       = X,
    y       = y,
    canais  = np.array(cols_presentes_global, dtype=str)
)
tamanho_mb = SEGMENTS_PATH.stat().st_size / 1e6
print(f"  ✅ {SEGMENTS_PATH.name:<35} {tamanho_mb:.1f} MB  (X={X.shape}, y={y.shape})")

# ── Metadados ────────────────────────────────────────────────────────────────
df_meta.to_parquet(META_PATH, index=False)
print(f"  ✅ {META_PATH.name:<35} {META_PATH.stat().st_size/1e6:.1f} MB  ({len(df_meta):,} linhas)")

# ── Relatório textual ─────────────────────────────────────────────────────────
linhas_report = [
    "RELATÓRIO DE SEGMENTAÇÃO — ENTREGÁVEL 5",
    "=" * 60,
    f"Fonte de sinal  : bronze/filtered_parquet/ (500 Hz, pré-filtrado Mendeley)",
    f"Normalização    : Z-score por paciente por canal (replicando PP4)",
    f"SQI             : kurtose > {KURT_THRESH} ou pp/std > {PP_THRESH} → rejeitar",
    f"Canais-chave SQI: {CANAIS_CHAVE}",
    f"Janela          : {WINDOW_S}s ({WINDOW_N} amostras @ {FS}Hz)",
    f"Overlap         : 50% ({HOP_N} amostras de avanço)",
    "",
    "RESULTADO GLOBAL",
    "-" * 60,
    f"  Total de janelas aprovadas : {len(all_windows):,}",
    f"  label=0 (Normal)           : {n_normal:,}  ({n_normal/len(y)*100:.1f}%)",
    f"  label=1 (FoG)              : {n_fog:,}  ({n_fog/len(y)*100:.1f}%)",
    f"  label=NaN                  : {n_nan:,}  ({n_nan/len(y)*100:.1f}%)",
    f"  Shape do array X           : {X.shape}",
    f"  Canais                     : {n_canais_global}",
    "",
    "VALIDAÇÃO",
    "-" * 60,
    f"  Mediana std intra-janela   : {np.nanmedian(std_intra):.4f}  (esperado ≈ 1.0)",
    f"  Mediana var inter-janela   : {np.nanmedian(var_inter):.4f}  (esperado < 0.1)",
    f"  Janelas com std intra ≈ 0  : {n_travadas}",
    "",
    "POR PACIENTE",
    "-" * 60,
]
for pid in sorted(df_meta['patient_id'].unique()):
    sub = df_meta[df_meta['patient_id'] == pid]
    fog  = int((sub['label'] == 1.0).sum())
    norm = int((sub['label'] == 0.0).sum())
    linhas_report.append(
        f"  {pid}: {len(sub):,} janelas  (normal={norm:,}, fog={fog:,})"
    )

linhas_report += [
    "",
    "SAÍDAS",
    "-" * 60,
    f"  segments.npz            → X (n_janelas,500,n_canais) + y (n_janelas,)",
    f"  windows_metadata.parquet → metadados por janela",
    "",
    "PRÓXIMO PASSO",
    "-" * 60,
    "  extração_2_features.ipynb",
    "  → Extrair features de cada janela do array 3D",
    "  → Features espectrais (band power), temporais (RMS, ZCR) e IMU",
]

REPORT_PATH.write_text("\n".join(linhas_report), encoding='utf-8')
print(f"  ✅ {REPORT_PATH.name:<35} texto")

In [ ]:
# Verificação: recarregar e confirmar integridade
dados = np.load(SEGMENTS_PATH, allow_pickle=True)
X_check = dados['X']
y_check = dados['y']
canais_check = dados['canais'].tolist()

print("Verificação do arquivo salvo:")
print(f"  X shape    : {X_check.shape}  ← esperado {X.shape}  {'✅' if X_check.shape == X.shape else '❌'}")
print(f"  y shape    : {y_check.shape}   ← esperado {y.shape}  {'✅' if y_check.shape == y.shape else '❌'}")
print(f"  n_canais   : {len(canais_check)}")
print(f"  X NaN%     : {np.isnan(X_check).mean()*100:.2f}%")
print(f"  y unique   : {np.unique(y_check[~np.isnan(y_check)])}")

## 12. Resumo Final

In [ ]:
print("=" * 60)
print("EXTRAÇÃO 1 — SEGMENTAÇÃO — RESUMO")
print("=" * 60)
print()
print("Entrada:")
print(f"  bronze/filtered_parquet/ — dados pré-filtrados do Mendeley")
print(f"  12 pacientes (incluindo 008 com 2 sessões)")
print()
print("Processamento:")
print(f"  Z-score por paciente por canal (replicando PP4)")
print(f"  SQI inline (kurtose + pp/std nos canais-chave)")
print(f"  Janelas de {WINDOW_S}s ({WINDOW_N} amostras) com {int(100*(1-HOP_N/WINDOW_N))}% de sobreposição")
print(f"  Janelas de transição (cross-task) descartadas")
print()
print("Saídas geradas:")
for path, desc in [
    (SEGMENTS_PATH, f"X {X.shape} + y {y.shape}"),
    (META_PATH,     f"{len(df_meta):,} janelas com metadados"),
    (REPORT_PATH,   "relatório textual"),
]:
    existe = '✅' if path.exists() else '❌'
    print(f"  {existe} {path.name:<35} {desc}")
print()
print("Validação:")
print(f"  Std intra-janela mediano  : {np.nanmedian(std_intra):.4f}  (esperado ≈ 1)")
print(f"  Var inter-janela mediana  : {np.nanmedian(var_inter):.4f}  (esperado < 0.1)")
print(f"  Janelas com std ≈ 0       : {n_travadas}  (segmento constante / sensor travado)")
print()
print("Distribuição de labels:")
print(f"  Normal (0): {n_normal:,}  ({n_normal/len(y)*100:.1f}%)")
print(f"  FoG (1)   : {n_fog:,}  ({n_fog/len(y)*100:.1f}%)")
print(f"  NaN       : {n_nan:,}  ({n_nan/len(y)*100:.1f}%)")
print()
print("Próximo passo:")
print("  extração_2_features.ipynb")
print("  → Extrair features espectrais, temporais e de IMU de cada janela")
print("=" * 60)
print("✅ Parte 1 (Segmentação) concluída.")
print("=" * 60)